# bench_med — Semi-supervised retinal OCT classification

We reuse the shared benchmarking and semi-supervised utilities to study
Retinal OCT disease classification. Training uses the Kermany 2018 dataset, while
out-of-distribution evaluation relies on OCT C8. The workflow is split into:

1. Supervised baselines via `BenchmarkRunner`
2. A Mean Teacher / Pseudo-Label experiment powered by `ss_vision_models.py`


## 1. Setup
Install optional dependencies (`timm`, `kaggle`) required by the vision registry.


In [ ]:
%%capture
!pip install -q timm==1.0.9 kaggle rich

## 2. Imports and reproducibility helpers
The notebook only uses models/augmentations defined in the repository.


In [ ]:
import os
from itertools import cycle
from pathlib import Path
from typing import Dict, Iterable

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

from pipelines_torch.benchmark import BenchmarkRunner
from pipelines_torch.vision_models import MODEL_REGISTRY as VISION_MODELS
from pipelines_torch.ss_vision_models import MeanTeacher, PseudoLabel
from pipelines_torch.base import SimplePredictor
from utils.metrics import accuracy_score, f1_score, precision_score, recall_score
from utils.utils import load_model

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 3. Download datasets
Both Kermany OCT2017 and OCT C8 live on Kaggle. Authentication is required.


In [ ]:
DATA_ROOT = Path("data_oct")
KERMANY_DATASET = "paultimothymooney/kermany2018"
OCTC8_DATASET = "obulisainaren/retinal-oct-c8"
KERMANY_DIR = DATA_ROOT / "kermany2018"
OCTC8_DIR = DATA_ROOT / "oct_c8"
DATA_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    if not KERMANY_DIR.exists():
        api.dataset_download_files(KERMANY_DATASET, path=KERMANY_DIR, unzip=True)
        print("✅ Downloaded Kermany 2018")
    if not OCTC8_DIR.exists():
        api.dataset_download_files(OCTC8_DATASET, path=OCTC8_DIR, unzip=True)
        print("✅ Downloaded OCT C8")
except Exception as exc:  # pragma: no cover
    print(f"⚠️ Kaggle download skipped: {exc}")


## 4. Datasets and transforms
We follow the ImageNet preprocessing pipeline required by the timm backbones.
A labelled subset is carved out for the supervised benchmark, while the remainder
is treated as unlabeled during SSL.


In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

weak_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

strong_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomApply([
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
        transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    ], p=0.8),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_dir = KERMANY_DIR / "OCT2017" / "train"
val_dir = KERMANY_DIR / "OCT2017" / "test"
if not train_dir.exists():
    raise FileNotFoundError("Expected OCT2017/train under the extracted Kermany dataset")

train_dataset = datasets.ImageFolder(train_dir, transform=weak_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
class_names = train_dataset.classes
num_classes = len(class_names)
print(f"Classes: {class_names}")


In [ ]:
def imagefolder_to_numpy(dataset: datasets.ImageFolder, batch_size: int = 32):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    xs, ys = [], []
    for xb, yb in loader:
        xs.append(xb)
        ys.append(yb)
    X = torch.cat(xs).numpy()
    y = torch.cat(ys).numpy()
    return X.astype(np.float32), y

rng = np.random.default_rng(SEED)
indices = np.arange(len(train_dataset))
rng.shuffle(indices)
label_fraction = 0.2
label_cutoff = int(len(indices) * label_fraction)
labeled_indices = indices[:label_cutoff]

labeled_subset = Subset(datasets.ImageFolder(train_dir, transform=val_transform), labeled_indices)
X_labeled, y_labeled = imagefolder_to_numpy(labeled_subset)
X_val_sup, y_val_sup = imagefolder_to_numpy(val_dataset)
print(f"Supervised subset: {X_labeled.shape}, Validation: {X_val_sup.shape}")


## 5. Supervised baseline with `BenchmarkRunner`
We reuse the same timm registry entries as the vision benchmark.


In [ ]:
metrics = [accuracy_score, f1_score, precision_score, recall_score]
vision_backbones = [
    "timm_convnextv2_tiny",
    "timm_efficientnetv2_s",
    "timm_swinv2_small",
]
model_configs = [
    {
        "name": name,
        "class": VISION_MODELS[name],
        "params": {"num_classes": num_classes},
    }
    for name in vision_backbones
]

runner = BenchmarkRunner(
    model_configs=model_configs,
    augmentations=[None],
    metrics=metrics,
    task_type="classification",
    device=DEVICE,
    epochs=5,
    batch_size=24,
    use_kfold=False,
    learning_rate=3e-4,
    path_start="bench_med_supervised",
    random_state=SEED,
)
supervised_results = runner.run(X_labeled, y_labeled)
supervised_results


### Reload checkpoints on the OCT2017 validation set


In [ ]:
sup_val_metrics = evaluate_saved_models(vision_backbones, X_val_sup, y_val_sup)
sup_val_metrics.sort_values("f1_macro", ascending=False)


## 6. Prepare loaders for semi-supervised training
The unlabeled pool consists of the remaining Kermany images (80% of the training
set). We also remap OCT C8 classes to the same label indices as Kermany.


In [ ]:
unlabeled_indices = indices[label_cutoff:]

labeled_dataset = Subset(datasets.ImageFolder(train_dir, transform=weak_transform), labeled_indices)
unlabeled_dataset_weak = Subset(datasets.ImageFolder(train_dir, transform=weak_transform), unlabeled_indices)
unlabeled_dataset_strong = Subset(datasets.ImageFolder(train_dir, transform=strong_transform), unlabeled_indices)

BATCH_L = 24
BATCH_U = 48
VAL_BATCH = 64

labeled_loader = DataLoader(labeled_dataset, batch_size=BATCH_L, shuffle=True, num_workers=2)
unlabeled_loader_weak = DataLoader(unlabeled_dataset_weak, batch_size=BATCH_U, shuffle=True, drop_last=True, num_workers=2)
unlabeled_loader_strong = DataLoader(unlabeled_dataset_strong, batch_size=BATCH_U, shuffle=True, drop_last=True, num_workers=2)
val_loader = DataLoader(datasets.ImageFolder(val_dir, transform=val_transform), batch_size=VAL_BATCH, shuffle=False, num_workers=2)


In [ ]:
class RemappedImageFolder(datasets.ImageFolder):
    def __init__(self, root: Path, transform, target_map: Dict[str, int]):
        super().__init__(root, transform=transform)
        filtered_samples = []
        for path, label in self.samples:
            cls_name = self.classes[label]
            if cls_name in target_map:
                filtered_samples.append((path, target_map[cls_name]))
        self.samples = filtered_samples
        self.targets = [t for _, t in filtered_samples]
        self.classes = list(target_map.keys())


octc8_map = {cls: idx for idx, cls in enumerate(class_names)}
octc8_dataset = RemappedImageFolder(OCTC8_DIR, transform=val_transform, target_map=octc8_map)
octc8_loader = DataLoader(octc8_dataset, batch_size=VAL_BATCH, shuffle=False, num_workers=2)
print(f"OCT C8 evaluation samples: {len(octc8_dataset)}")


## 7. Semi-supervised training loop
We rely on `MeanTeacher` / `PseudoLabel` from `ss_vision_models.py`. The training
loop simply calls the shared `ssl_loss` method on each batch.


In [ ]:
def build_ssl_model(backbone_name: str = "timm_convnextv2_tiny", *, use_mean_teacher: bool = True) -> nn.Module:
    if backbone_name not in VISION_MODELS:
        raise KeyError(f"Unknown backbone {backbone_name}")
    backbone = VISION_MODELS[backbone_name](num_classes=num_classes)
    if use_mean_teacher:
        model = MeanTeacher(backbone, unsup_weight=1.0, rampup=5, ema_decay=0.996)
    else:
        model = PseudoLabel(backbone, threshold=0.95, unsup_weight=1.0, rampup=5)
    return model.to(DEVICE)


def evaluate_loader(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            preds = logits.argmax(dim=1).cpu().numpy()
            y_true.append(yb.numpy())
            y_pred.append(preds)
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred)),
        "recall_macro": float(recall_score(y_true, y_pred)),
    }


def train_ssl(backbone_name: str = "timm_convnextv2_tiny", *, epochs: int = 8, lr: float = 3e-4, use_mean_teacher: bool = True):
    model = build_ssl_model(backbone_name, use_mean_teacher=use_mean_teacher)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    unlabeled_cycle = cycle(zip(unlabeled_loader_weak, unlabeled_loader_strong))
    for epoch in range(epochs):
        model.train()
        logs_epoch = []
        for xb_l, yb_l in labeled_loader:
            try:
                (xb_u_w, _), (xb_u_s, _) = next(unlabeled_cycle)
            except StopIteration:
                unlabeled_cycle = cycle(zip(unlabeled_loader_weak, unlabeled_loader_strong))
                (xb_u_w, _), (xb_u_s, _) = next(unlabeled_cycle)
            xb_l = xb_l.to(DEVICE)
            yb_l = yb_l.to(DEVICE)
            xb_u_w = xb_u_w.to(DEVICE)
            xb_u_s = xb_u_s.to(DEVICE)

            optimizer.zero_grad()
            loss_sup, loss_unsup, _ = model.ssl_loss((xb_l, yb_l), (xb_u_w, None), epoch)
            logits_strong = model(xb_u_s)
            loss = loss_sup + loss_unsup
            loss.backward()
            optimizer.step()
            if hasattr(model, "update_teacher"):
                model.update_teacher()
            logs_epoch.append({"sup": float(loss_sup.item()), "unsup": float(loss_unsup.item())})
        eval_metrics = evaluate_loader(model, val_loader)
        eval_metrics.update({
            "loss_sup": float(np.mean([l["sup"] for l in logs_epoch])),
            "loss_unsup": float(np.mean([l["unsup"] for l in logs_epoch])),
        })
        history.append(eval_metrics)
        print(f"Epoch {epoch+1}/{epochs} — val F1: {eval_metrics['f1_macro']:.3f}")
    return model, pd.DataFrame(history)


### Run Mean Teacher on the semi-supervised split


In [ ]:
ssl_model, ssl_history = train_ssl("timm_convnextv2_tiny", epochs=8, lr=3e-4, use_mean_teacher=True)
ssl_history


### Evaluate on OCT2017 validation and OCT C8


In [ ]:
val_ssl_metrics = evaluate_loader(ssl_model, val_loader)
octc8_metrics = evaluate_loader(ssl_model, octc8_loader)
print("Validation:", val_ssl_metrics)
print("OCT C8:", octc8_metrics)


## 8. Next steps
- Switch `use_mean_teacher=False` in `train_ssl` to run the Pseudo-Label variant.
- Increase the labelled fraction to study how performance scales with supervision.
- Swap in alternative backbones from `vision_models.py` for a more thorough grid.
